### Assignment : Week 1 
## Modeling simple RL problems by making their MDPs in Python

We will create the MDPs for some of the example problems from Grokking textbook. For the simple environments, we can just hardcode the MDPs into a dictionary by exhaustively encoding the whole state space and the transition function. We will also go through a more complicated example where the state space is too large to be manually coded and we need to implement the transition function based on some state parameters.

Later on, you will not need to implement the MDPs of common RL problems yourself, most of the work is already done by the OpenAI Gym library, which includes models for most of the famous RL envis.

You can start this assignment during/after reading Grokking Ch-2.

In [16]:
pip install gymnasium


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Environment 0 - Bandit Walk

Let us consider the BW environment on Page 39. 

State Space has 3 elements, states 0, 1 and 2.
States 0 and 2 are terminal states and state 1 is the starting state.

Action space has 2 elements, left and right.

The environment is deterministic - transition probability of any action is 1.

Only 1 (State, Action, State') tuple has positive reward, (1, Right, 2) gives the agent +1 reward.

We'll model this MDP as a dictionary. This code is an example for the upcoming exercises.

In [17]:
bw_mdp = {

    0 : {
        "Right" : [(1, 0, 0, True)],
        "Left" : [(1, 0, 0, True)]
    },

    1 : {
        "Right" : [(1, 2, 1, True)],
        "Left" : [(1, 0, 0, True)]
    },

    2 : {
        "Right" : [(1, 2, 0, True)],
        "Left" : [(1, 2, 0, True)]
    }
    
}

Note that by convention, all actions from terminal states still lead to the same state with reward 0.

## Environment 1 - Slippery Walk

Let us now look at the BSW environment on Page 40. We'll model a slightly modified version of BSW with 7 states instead (i.e the SWF envi on Page 67). It will be useful in the coming weeks.

Here, states 0 and 6 are terminal states and state 3 is the starting state.

Action space has again 2 elements, left and right.

The environment is now stochastic, transition probability of any action is as follows -
If agent chooses `Right` at a non-terminal state,
- $50\%$ times it will go to `Right` state
- $33\frac{1}{3} \%$ times it will stay in same state
- $16\frac{2}{3}\%$ times it will go to `Left`state

This time, 2 different (State, Action, State') tuples have positive rewards, you need to find them.

We'll again model this MDP as a dictionary. Part of the code is written for you.

In [18]:
swf_mdp = {}
p_right=0.5
p_stay=1/3
p_left=1/6
reward_goal=1
reward_hole=0
terminal=[0,6]

for s in range(7):
    swf_mdp[s]={}
    
    if s in terminal:
        swf_mdp[s]["Right"]=[(1.0,s,reward_goal if s==6 else reward_hole,True)]
        swf_mdp[s]["Left"]=[(1.0,s,reward_goal if s==6 else reward_hole,True)]
        continue
    
    next_right = min(s+1, 6)
    next_left  = max(s-1, 0)
    swf_mdp[s]["Right"] = [
        (p_right, next_right, reward_goal if next_right==6 else reward_hole, next_right in terminal),
        (p_stay, s, reward_goal if s==6 else reward_hole, s in terminal),
        (p_left, next_left, reward_goal if next_left==6 else reward_hole, next_left in terminal),
    ]
    
    swf_mdp[s]["Left"] = [
        (p_right, next_left, reward_goal if next_left==6 else reward_hole, next_left in terminal),
        (p_stay, s, reward_goal if s==6 else reward_hole, s in terminal),
        (p_left, next_right, reward_goal if next_right==6 else reward_hole, next_right in terminal),
    ]
import pprint
pprint.pprint(swf_mdp)


{0: {'Left': [(1.0, 0, 0, True)], 'Right': [(1.0, 0, 0, True)]},
 1: {'Left': [(0.5, 0, 0, True),
              (0.3333333333333333, 1, 0, False),
              (0.16666666666666666, 2, 0, False)],
     'Right': [(0.5, 2, 0, False),
               (0.3333333333333333, 1, 0, False),
               (0.16666666666666666, 0, 0, True)]},
 2: {'Left': [(0.5, 1, 0, False),
              (0.3333333333333333, 2, 0, False),
              (0.16666666666666666, 3, 0, False)],
     'Right': [(0.5, 3, 0, False),
               (0.3333333333333333, 2, 0, False),
               (0.16666666666666666, 1, 0, False)]},
 3: {'Left': [(0.5, 2, 0, False),
              (0.3333333333333333, 3, 0, False),
              (0.16666666666666666, 4, 0, False)],
     'Right': [(0.5, 4, 0, False),
               (0.3333333333333333, 3, 0, False),
               (0.16666666666666666, 2, 0, False)]},
 4: {'Left': [(0.5, 3, 0, False),
              (0.3333333333333333, 4, 0, False),
              (0.16666666666666666, 5,

Feel free to automate filling this MDP, but ensure that it is correctly filled as it'll be back in next week's assignment.

## Environment 2 - Frozen Lake Environment

This environment is described on Page 46.

The FL environment has a large state space, so it's better to generate most of the MDP via Python instead of typing stuff manually.

Note that all 5 states - 5, 7, 11, 12, 15 are terminal states, so keep that in mind while constructing the MDP.

There are 4 actions now - Up, Down, Left, Right.

The environment is stochastic, and states at the border of lake will require separate treatment.



Yet again we will model this MDP as a (large) dictionary.

In [19]:
def to_state(r, c):
    return r*4+c
def in_bounds(r, c):
    return 0<= r<4 and 0<=c<4
def is_terminal(cell):
    return cell in "HG"

grid=["SFFF",
      "FHFH",
      "FFFH",
      "HFFG"]
ACTIONS={
    0:(0,-1),   #Left
    1:(1,0),    #Down
    2:(0,1),    #Right
    3:(-1,0),   #Up
}

LEFT = {0: 3, 1: 0, 2: 1, 3: 2}
RIGHT = {0: 1, 1: 2, 2: 3, 3: 0}

P[s]={}
for r in range(4):
    for c in range(4):
        s=to_state(r,c)
        cell=grid[r][c]
        if is_terminal(cell):
            for a in range(4):
                P[s][a]=[(1.0, s, 0.0, True)]
        else:
            for a in range(4):
                P[s][a] = []
                for slip_a in [a,LEFT[a],RIGHT[a]]:
                    dr,dc=ACTIONS[slip_a]
                    nr,nc=r+dr,c+dc

                    if not in_bounds(nr, nc):
                        ns=s
                    else:
                        ns=to_state(nr, nc)

                    next_cell = grid[nr][nc] if in_bounds(nr, nc) else cell
                    done = is_terminal(next_cell)
                    reward = 1.0 if next_cell == "G" else 0.0

                    P[s][a].append((1/3, ns, reward, done))



You might need to do some stuff manually, but make sure to automate most of it.

You can check your implementation of the FL environment by comparing it with the one in OpenAI Gym.

You don't need to worry about Gym right now, we'll set it up in the coming weeks. But here is the code to import an MDP.

In [20]:
import gymnasium as gym

env=gym.make("FrozenLake-v1")
P2=env.unwrapped.P


In [21]:
pprint.pprint(P)

{0: {0: [(0.3333333333333333, 0, 0.0, False),
         (0.3333333333333333, 0, 0.0, False),
         (0.3333333333333333, 4, 0.0, False)],
     1: [(0.3333333333333333, 4, 0.0, False),
         (0.3333333333333333, 0, 0.0, False),
         (0.3333333333333333, 1, 0.0, False)],
     2: [(0.3333333333333333, 1, 0.0, False),
         (0.3333333333333333, 4, 0.0, False),
         (0.3333333333333333, 0, 0.0, False)],
     3: [(0.3333333333333333, 0, 0.0, False),
         (0.3333333333333333, 1, 0.0, False),
         (0.3333333333333333, 0, 0.0, False)]},
 1: {0: [(0.3333333333333333, 0, 0.0, False),
         (0.3333333333333333, 1, 0.0, False),
         (0.3333333333333333, 5, 0.0, True)],
     1: [(0.3333333333333333, 5, 0.0, True),
         (0.3333333333333333, 0, 0.0, False),
         (0.3333333333333333, 2, 0.0, False)],
     2: [(0.3333333333333333, 2, 0.0, False),
         (0.3333333333333333, 5, 0.0, True),
         (0.3333333333333333, 1, 0.0, False)],
     3: [(0.3333333333333333,

Since the imported MDP is also just a dictionary, we can just print it.

In [22]:
pprint.pprint(P2)

{0: {0: [(0.33333333333333337, 0, 0, False),
         (0.3333333333333333, 0, 0, False),
         (0.33333333333333337, 4, 0, False)],
     1: [(0.33333333333333337, 0, 0, False),
         (0.3333333333333333, 4, 0, False),
         (0.33333333333333337, 1, 0, False)],
     2: [(0.33333333333333337, 4, 0, False),
         (0.3333333333333333, 1, 0, False),
         (0.33333333333333337, 0, 0, False)],
     3: [(0.33333333333333337, 1, 0, False),
         (0.3333333333333333, 0, 0, False),
         (0.33333333333333337, 0, 0, False)]},
 1: {0: [(0.33333333333333337, 1, 0, False),
         (0.3333333333333333, 0, 0, False),
         (0.33333333333333337, 5, 0, True)],
     1: [(0.33333333333333337, 0, 0, False),
         (0.3333333333333333, 5, 0, True),
         (0.33333333333333337, 2, 0, False)],
     2: [(0.33333333333333337, 5, 0, True),
         (0.3333333333333333, 2, 0, False),
         (0.33333333333333337, 1, 0, False)],
     3: [(0.33333333333333337, 2, 0, False),
         (0.